# Template Translation Detection of Egyptian Articles 

## Supervised Classification Algorithms:

### Logistic Regression:

In [ ]:
import warnings
import pandas as pd
warnings.filterwarnings("ignore")

df = pd.read_csv("../../Experimental-Setups/arzwiki-20240101-all-20k.csv")
df

#### Input Features:

In [ ]:
import json
import numpy as np
import pandas as pd

def fetch_embeddings(page_text, dataframe_name, embeddings_file):
    idx = np.where(dataframe_name["page_text_cleaned"] == page_text)[0][0]
    page_title = dataframe_name['page_title'][idx]
    return embeddings_file[page_title]
    
embeddings_files = ["../../Experimental-Setups/arzwiki-20240101-all-20k-sparknlp-embeddings.json", 
                   "../../Experimental-Setups/arzwiki-20240101-all-20k-camelbert-embeddings.json"]

for file in embeddings_files:
    with open(file) as json_file:
        fetched_page_text_embeddings = json.load(json_file)
        df[f'{file.split("/")[-1].split("-")[-2]}_page_text_embeddings'] = df['page_text_cleaned'].apply(lambda page_text: fetch_embeddings(page_text, df, fetched_page_text_embeddings))

df['label'] = df['label'].map({'template-translated': 1, 'human-generated': 0})

df

#### Ablations:

In [ ]:
import pandas as pd

def prepare_meta_features(df, features):
    if features == "metadata1":
        X = []
        for i in range(df.shape[0]):
            x = []
            x.append(df['total_edits'][i])    # 1
            X.append(x) 
        y = df['label'].to_list()
        return X, y

    elif features == "metadata2":
        X = []
        for i in range(df.shape[0]):
            x = []
            x.append(df['total_editors'][i])  # 2
            X.append(x) 
        y = df['label'].to_list()
        return X, y

    elif features == "metadata3":
        X = []
        for i in range(df.shape[0]):
            x = []
            x.append(df['total_bytes'][i])    # 3
            X.append(x) 
        y = df['label'].to_list()
        return X, y

    elif features == "metadata4":
        X = []
        for i in range(df.shape[0]):
            x = []
            x.append(df['total_chars'][i])    # 4
            X.append(x) 
        y = df['label'].to_list()
        return X, y

    elif features == "metadata5":
        X = []
        for i in range(df.shape[0]):
            x = []
            x.append(df['total_words'][i])    # 5
            X.append(x) 
        y = df['label'].to_list()
        return X, y

    elif features == "metadata1+2":
        X = []
        for i in range(df.shape[0]):
            x = []
            x.append(df['total_edits'][i])    # 1
            x.append(df['total_editors'][i])  # 2
            X.append(x) 
        y = df['label'].to_list()
        return X, y

    elif features == "metadata3+4+5":
        X = []
        for i in range(df.shape[0]):
            x = []
            x.append(df['total_bytes'][i])    # 3
            x.append(df['total_chars'][i])    # 4
            x.append(df['total_words'][i])    # 5
            X.append(x) 
        y = df['label'].to_list()
        return X, y

    elif features == "metadata_all":
        X = []
        for i in range(df.shape[0]):
            x = []
            x.append(df['total_edits'][i])    # 1
            x.append(df['total_editors'][i])  # 2
            x.append(df['total_bytes'][i])    # 3
            x.append(df['total_chars'][i])    # 4
            x.append(df['total_words'][i])    # 5
            X.append(x) 
        y = df['label'].to_list()
        return X, y

    else: 
        print("Error: Choose Features: Metadata 1 to 5, Metadata 1+2, Metadata 3+4+5, Metadata All")
        pass

In [ ]:
import seaborn as sns
from sklearn import metrics
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score, cross_val_predict

features = ["metadata1", "metadata2", "metadata3", "metadata4", "metadata5", "metadata1+2", "metadata3+4+5", "metadata_all"]

for feature in features:

    print(f"##### FEATURE={feature.upper()} #####\n")
    X, y = prepare_meta_features(df, features=feature)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.20, stratify=y, shuffle=True, random_state=2024)
    
    logistic_regression_classifier = LogisticRegression()
    
    y_trained = cross_val_score(logistic_regression_classifier, X_train, y_train, cv=5)
    print("Training Accuracy: %0.2f (+/- %0.2f)" % (y_trained.mean()*100, y_trained.std() * 2), "%")
    
    y_predicted = cross_val_predict(logistic_regression_classifier, X_test, y_test, cv=5)
    print("Testing Accuracy: %0.2f (+/- %0.2f)" %  (metrics.accuracy_score(y_test, y_predicted)*100, y_predicted.std() * 2), "%")
    
    print("\nClassification Report:\n", classification_report(y_test, y_predicted), "\n")
    
    cm = confusion_matrix(y_test, y_predicted)
    sns.heatmap(cm, annot=True, cmap='Blues')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(f'Confusion Matrix of Logistic Regression\nf={feature}')
    plt.savefig(f"plots/lr_cm_{feature}.png", bbox_inches='tight', dpi=100, facecolor='white', transparent=False)
    plt.show()
    
    print("\n")
    
    logistic_regression_classifier.fit(X_train, y_train)
    logistic_regression_classifier.predict_proba(X_test)[:, 1]
    
    false_positive, true_positive, thresholds = roc_curve(y_test, y_predicted)
    auc_roc = auc(false_positive, true_positive)
    
    fig, ax = plt.subplots()
    ax.plot(false_positive, true_positive, label='ROC Curve (Area = %0.2f)' % auc_roc)
    ax.set_xlabel('False Positive Rate (FPR)')
    ax.set_ylabel('True Positive Rate (TPR)')
    ax.set_title(f'ROC Curve of Logistic Regression\nf={feature}')
    plt.savefig(f"plots/lr_roc_{feature}.png", bbox_inches='tight', dpi=100, facecolor='white', transparent=False)
    ax.legend()
    plt.show()

    print("\n")

#### Results:

In [ ]:
import numpy as np
import pandas as pd

def prepare_features(df, features, model=None):
    if features == "metadata":
        X = []
        for i in range(df.shape[0]):
            x = []
            x.append(df['total_edits'][i])
            x.append(df['total_editors'][i])
            x.append(df['total_bytes'][i])
            x.append(df['total_chars'][i])
            x.append(df['total_words'][i])
            X.append(x) # Only page_metadata
        y = df['label'].to_list()
        return X, y

    elif features == "embeddings":
        if model == "sparknlp":
            X = []
            for i in range(df.shape[0]):
                x = []
                X.append(df['sparknlp_page_text_embeddings'].to_list()[i]) # Only page_text_embeddings
            y = df['label'].to_list()
            return X, y
        
        elif model == "camelbert":
            X = []
            for i in range(df.shape[0]):
                x = []
                X.append(df['camelbert_page_text_embeddings'].to_list()[i]) # Only page_text_embeddings
            y = df['label'].to_list()
            return X, y
        
        else: 
            print("Error: Choose An Embedding Model: SparkNLP or CAMeLBERT")
            pass
            
    elif features == "metadata+embeddings":
        if model == "sparknlp":
            X = []
            for i in range(df.shape[0]):
                x = []
                x.append(df['total_edits'][i])
                x.append(df['total_editors'][i])
                x.append(df['total_bytes'][i])
                x.append(df['total_chars'][i])
                x.append(df['total_words'][i])
                X.append(np.hstack([x, df['sparknlp_page_text_embeddings'].to_list()[i]])) # Both page_metadata + page_text_embeddings
            y = df['label'].to_list()
            return X, y
        
        elif model == "camelbert":
            X = []
            for i in range(df.shape[0]):
                x = []
                x.append(df['total_edits'][i])
                x.append(df['total_editors'][i])
                x.append(df['total_bytes'][i])
                x.append(df['total_chars'][i])
                x.append(df['total_words'][i])
                X.append(np.hstack([x, df['camelbert_page_text_embeddings'].to_list()[i]])) # Both page_metadata + page_text_embeddings
            y = df['label'].to_list()
            return X, y
  
    else: 
        print("Error: Choose Features: Metadata, Embeddings, or Metadata+Embeddings")
        pass

In [ ]:
import seaborn as sns
from sklearn import metrics
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score, cross_val_predict

feature, model = "metadata", "none"

print(f"##### FEATURE={feature.upper()}, MODEL={model.upper()} #####\n")

X, y = prepare_features(df, features=feature, model=model)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.20, stratify=y, shuffle=True, random_state=2024)

logistic_regression_classifier = LogisticRegression()

y_trained = cross_val_score(logistic_regression_classifier, X_train, y_train, cv=5)
print("Training Accuracy: %0.2f (+/- %0.2f)" % (y_trained.mean()*100, y_trained.std() * 2), "%")

y_predicted = cross_val_predict(logistic_regression_classifier, X_test, y_test, cv=5)
print("Testing Accuracy: %0.2f (+/- %0.2f)" %  (metrics.accuracy_score(y_test, y_predicted)*100, y_predicted.std() * 2), "%")

print("\nClassification Report:\n", classification_report(y_test, y_predicted), "\n")

cm = confusion_matrix(y_test, y_predicted)
sns.heatmap(cm, annot=True, cmap='Blues')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title(f'Confusion Matrix of Logistic Regression\nf={feature}, m={model}')
plt.savefig(f"plots/lr_cm_{feature}_{model}.png", bbox_inches='tight', dpi=100, facecolor='white', transparent=False)
plt.show()

print("\n")

logistic_regression_classifier.fit(X_train, y_train)
logistic_regression_classifier.predict_proba(X_test)[:, 1]

false_positive, true_positive, thresholds = roc_curve(y_test, y_predicted)
auc_roc = auc(false_positive, true_positive)

fig, ax = plt.subplots()
ax.plot(false_positive, true_positive, label='ROC Curve (Area = %0.2f)' % auc_roc)
ax.set_xlabel('False Positive Rate (FPR)')
ax.set_ylabel('True Positive Rate (TPR)')
ax.set_title(f'ROC Curve of Logistic Regression\nf={feature}, m={model}')
plt.savefig(f"plots/lr_roc_{feature}_{model}.png", bbox_inches='tight', dpi=100, facecolor='white', transparent=False)
ax.legend()
plt.show()

print("\n")

features = ["embeddings", "metadata+embeddings"]
models = ["sparknlp", "camelbert"]

for feature in features:

    for model in models:

        print(f"##### FEATURE={feature.upper()}, MODEL={model.upper()} #####\n")
        X, y = prepare_features(df, features=feature, model=model)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.20, stratify=y, shuffle=True, random_state=2024)
        
        logistic_regression_classifier = LogisticRegression()
        
        y_trained = cross_val_score(logistic_regression_classifier, X_train, y_train, cv=5)
        print("Training Accuracy: %0.2f (+/- %0.2f)" % (y_trained.mean()*100, y_trained.std() * 2), "%")
        
        y_predicted = cross_val_predict(logistic_regression_classifier, X_test, y_test, cv=5)
        print("Testing Accuracy: %0.2f (+/- %0.2f)" %  (metrics.accuracy_score(y_test, y_predicted)*100, y_predicted.std() * 2), "%")
        
        print("\nClassification Report:\n", classification_report(y_test, y_predicted), "\n")
        
        cm = confusion_matrix(y_test, y_predicted)
        sns.heatmap(cm, annot=True, cmap='Blues')
        plt.xlabel('Predicted Label')
        plt.ylabel('True Label')
        plt.title(f'Confusion Matrix of Logistic Regression\nf={feature}, m={model}')
        plt.savefig(f"plots/lr_cm_{feature}_{model}.png", bbox_inches='tight', dpi=100, facecolor='white', transparent=False)
        plt.show()
        
        print("\n")
        
        logistic_regression_classifier.fit(X_train, y_train)
        logistic_regression_classifier.predict_proba(X_test)[:, 1]
        
        false_positive, true_positive, thresholds = roc_curve(y_test, y_predicted)
        auc_roc = auc(false_positive, true_positive)
        
        fig, ax = plt.subplots()
        ax.plot(false_positive, true_positive, label='ROC Curve (Area = %0.2f)' % auc_roc)
        ax.set_xlabel('False Positive Rate (FPR)')
        ax.set_ylabel('True Positive Rate (TPR)')
        ax.set_title(f'ROC Curve of Logistic Regression\nf={feature}, m={model}')
        plt.savefig(f"plots/lr_roc_{feature}_{model}.png", bbox_inches='tight', dpi=100, facecolor='white', transparent=False)
        ax.legend()
        plt.show()

        print("\n")